In [ ]:
import pandas as pd

from pyfaidx import Fasta

from Bio import SeqIO

import matplotlib.pyplot as plt

import numpy as np

import seaborn as sns


In [ ]:
circRNA_df = pd.read_csv("circExor/resources/circRNA subcellular localization information.txt",sep = '\t')

localization_df = circRNA_df


In [ ]:
circRNA_df


In [ ]:
           

score_counts = circRNA_df['RNALocate_Score'].value_counts()

print(score_counts)

                             

plt.figure(figsize=(8, 6))

plt.hist(circRNA_df['RNALocate_Score'], bins=20, range=(0.3, 1.0), color='skyblue', edgecolor='black')

         

plt.title('Distribution of RNALocate_Score before cutoff', fontsize=18)

plt.xlabel('RNALocate_Score', fontsize=16)

plt.ylabel('Frequency', fontsize=16)

      

plt.tight_layout()

plt.show()



In [ ]:
localization_counts = circRNA_df['Subcellular_Localization'].value_counts()

           

print(localization_counts)

       

plt.figure(figsize=(10, 6))

localization_counts.plot(kind='bar', color='skyblue', edgecolor='black')

         

plt.title('Distribution of Subcellular Localization before cutoff', fontsize=18)

                                                     

plt.ylabel('Frequency', fontsize=16)

      

plt.tight_layout()

plt.show()


In [ ]:
df_filtered = circRNA_df[circRNA_df['RNALocate_Score'] >= 0.30]

          

print(f'Rows remaining after truncation: {len(df_filtered)} rows')

           

plt.figure(figsize=(8, 6))

plt.hist(df_filtered['RNALocate_Score'], bins=100, range=(0, 1.0), color='skyblue', edgecolor='black')

         

plt.title('Distribution of RNALocate_Score after > 0.30 cutoff', fontsize=18)

plt.xlabel('RNALocate_Score', fontsize=16)

plt.ylabel('Frequency', fontsize=16)

      

plt.tight_layout()

plt.show()


In [ ]:
localization_counts = df_filtered['Subcellular_Localization'].value_counts()

           

print(localization_counts)

       

plt.figure(figsize=(10, 6))

localization_counts.plot(kind='bar', color='skyblue', edgecolor='black')

         

plt.title('Distribution of Subcellular Localization(score>=0.38)', fontsize=14)

plt.xlabel('Subcellular Localization', fontsize=12)

plt.ylabel('Frequency', fontsize=12)

      

plt.tight_layout()

plt.show()


In [ ]:
def process_rna_symbol(symbol):

                                   

    if symbol.startswith("novel_"):

        symbol = symbol[6:] 

    

                       

    if "hsa" not in symbol and "mmu" not in symbol and "exo" not in symbol:

        return None                    

    

                            

    parts = symbol.split('-')

    if len(parts) == 3 and parts[0] == "piR":

        symbol = f"hsa-piR-{parts[2]}" 

    

    return symbol


In [ ]:
localization_df['RNA_Symbol'] = localization_df['RNA_Symbol'].apply(process_rna_symbol)

localization_df = localization_df[localization_df['RNALocate_Score'] >= 0.30]

              

localization_df['tag'] = 0

               

exosome_locations = ['Extracellular exosome', 'Extracellular vesicle', 'Microvesicle']

                 

mask = localization_df['Subcellular_Localization'].isin(exosome_locations)

localization_df.loc[mask, 'tag'] = 1

                  

localization_df.dropna(subset=['RNA_Symbol'], inplace=True)


In [ ]:
fasta_files = [

    "circExor/references/circRNA/mouse_mm9_circRNAs_putative_spliced_sequence.fa",

    "circExor/references/circRNA/hsa_circbase_seq.fa",

    "circExor/references/circRNA/exo_circRNA_sequences.fa",

]

                                

def parse_fasta(file_path):

    seq_dict = {}

    for record in SeqIO.parse(file_path, "fasta"):

        name = record.id

        sequence = str(record.seq)

                            

        if "|" in name:         

            symbol = name.split("|")[0]

        elif "(" in name:         

            symbol = name.split("(")[0]

        else:         

            symbol = name

        seq_dict[symbol] = sequence

    return seq_dict

                            

rna_sequences = {}

for fasta in fasta_files:

    rna_sequences.update(parse_fasta(fasta))

         

                         

localization_df["Sequence"] = localization_df.iloc[:, 2].map(rna_sequences).fillna("Sequence_not_found")



In [ ]:
localization_df


In [ ]:
                                                          

original_row_count = len(localization_df)        

localization_df = localization_df[

    ~localization_df['Sequence'].str.contains('N', na=False)                

]

localization_df = localization_df[

    (localization_df['Sequence'] != "Sequence_not_found") &                                

    localization_df['Sequence'].notna()          

]

new_row_count = len(localization_df)         

         

deleted_rows = original_row_count - new_row_count

         

print(f"Deleted {deleted_rows} rows")


In [ ]:
                

localization_df['Sequence_Length'] = localization_df['Sequence'].str.len()

       

average_length = localization_df['Sequence_Length'].mean()

print(f"Average sequence length: {average_length:.2f}")

        

bins = np.histogram_bin_edges(localization_df['Sequence_Length'], bins=30)             

bin_counts, bin_edges = np.histogram(localization_df['Sequence_Length'], bins=bins)

             

print("Sequence length distribution by bins：")

for i in range(len(bin_counts)):

    print(f"{bin_edges[i]:.1f} - {bin_edges[i+1]:.1f}: {bin_counts[i]} sequences")

       

plt.figure(figsize=(10, 6))

sns.histplot(localization_df['Sequence_Length'], bins=bins, kde=True, color='skyblue', label='Length Distribution')

          

plt.axvline(average_length, color='red', linestyle='--', label=f'Mean: {average_length:.2f}')

      

plt.title('Sequence Length Distribution', fontsize=16)

plt.xlabel('Sequence Length', fontsize=12)

plt.ylabel('Frequency', fontsize=12)

plt.legend()

plt.grid(alpha=0.3)

plt.tight_layout()

      

plt.show()


In [ ]:
                

localization_df['Sequence_Length'] = localization_df['Sequence'].str.len()

original_row_count = len(localization_df)

                                               

localization_df = localization_df[localization_df['Sequence_Length'] <= 11238]

new_row_count = len(localization_df)

deleted_rows = original_row_count - new_row_count

print(f"Deleted {deleted_rows} rows")



In [ ]:
                

localization_df['Sequence_Length'] = localization_df['Sequence'].str.len()

             

max_length = localization_df['Sequence_Length'].max()

min_length = localization_df['Sequence_Length'].min()

average_length = localization_df['Sequence_Length'].mean()

print(f"Minimum sequence length: {min_length}")

print(f"Maximum sequence length: {max_length}")

print(f"Average sequence length: {average_length:.2f}")

        

bins = np.histogram_bin_edges(localization_df['Sequence_Length'], bins=30)             

bin_counts, bin_edges = np.histogram(localization_df['Sequence_Length'], bins=bins)

             

print("\nSequence length distribution by bins：")

for i in range(len(bin_counts)):

    print(f"{bin_edges[i]:.1f} - {bin_edges[i+1]:.1f}: {bin_counts[i]} sequences")

       

plt.figure(figsize=(10, 6))

sns.histplot(localization_df['Sequence_Length'], bins=bins, kde=True, color='skyblue', label='Length Distribution')

                  

plt.axvline(min_length, color='green', linestyle='--', label=f'Min: {min_length}')

plt.axvline(max_length, color='orange', linestyle='--', label=f'Max: {max_length}')

plt.axvline(average_length, color='red', linestyle='--', label=f'Mean: {average_length:.2f}')

      

plt.title('Sequence Length Distribution(score>=0.5)', fontsize=16)

plt.xlabel('Sequence Length', fontsize=12)

plt.ylabel('Frequency', fontsize=12)

plt.legend()

plt.grid(alpha=0.3)

plt.tight_layout()

      

plt.show()


In [ ]:
localization_counts = localization_df['Subcellular_Localization'].value_counts()

           

print(localization_counts)

       

plt.figure(figsize=(10, 6))

localization_counts.plot(kind='bar', color='skyblue', edgecolor='black')

         

plt.title('Distribution of Subcellular Localization(score>=0.5)', fontsize=14)

plt.xlabel('Subcellular Localization', fontsize=12)

plt.ylabel('Frequency', fontsize=12)

      

plt.tight_layout()

plt.show()


In [ ]:


localization_counts = localization_df['tag'].value_counts()

           

print(localization_counts)

       

plt.figure(figsize=(6, 6))

ax = localization_counts.plot(kind='bar', color='skyblue', edgecolor='black', width=0.5)

               

for p in ax.patches:

    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),

                ha='center', va='center', xytext=(0, 10), textcoords='offset points')

         

plt.title('Distribution of Subcellular Localization(score>=0.30)', fontsize=14)

plt.xlabel('Subcellular Localization', fontsize=12)

plt.ylabel('Frequency', fontsize=12)

                             

plt.xticks(rotation=45)

      

plt.tight_layout()

plt.show()


In [ ]:
output_csv = "./output_with_sequences.csv"           

localization_df.to_csv(output_csv, sep="\t", index=False)
